# Statistical Methods for Decision-Making

**Recruiter-facing end-to-end analysis · Statistical inference · Python 3.12/3.13**

> Shingle B is below the 0.35 limit by both t and Wilcoxon tests (t-test p=0.0021); Shingle A is not below the limit by the prespecified t-test (p=0.075).

## Executive summary

**Objective:** Turn wholesale, student-survey, and shingle measurements into assumption-aware business and quality decisions.

**Data:** 440 wholesale customers, 62 survey respondents, and 36 shingle rows across three CSV files.

**Verified result:** Shingle B is below the 0.35 limit by both t and Wilcoxon tests (t-test p=0.0021); Shingle A is not below the limit by the prespecified t-test (p=0.075).

**Decision supported:** Prioritize category/channel investigation and decide whether moisture evidence supports a specification claim.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A wholesale manager, survey analyst, or quality engineer.

**Decision:** Prioritize category/channel investigation and decide whether moisture evidence supports a specification claim.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '01-statistical-methods'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 01-statistical-methods


## 4. Data provenance and scope

Wholesale data aligns with the UCI Wholesale Customers dataset. Survey and shingle collection protocols and reuse terms were not supplied in the original project.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

                   file  size_mb           sha256
           shingles.csv    0.000 b9a12bc5aee1fb60
  university_survey.csv    0.005 4ff41af9f9795d3d
wholesale_customers.csv    0.020 0b893d36b4a7abce


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


shingles.csv: 2 columns
   A    B
0.44 0.14
0.61 0.15
0.47 0.31
0.30 0.16
0.15 0.37

university_survey.csv: 14 columns
 ID Gender  Age  Class      Major Grad Intention  ...  Salary Social Networking  Satisfaction  Spending  Computer  Text Messages
  1 Female   20 Junior      Other            Yes  ...      50                 1             3       350    Laptop            200
  2   Male   23 Senior Management            Yes  ...      25                 1             4       360    Laptop             50
  3   Male   21 Junior      Other            Yes  ...      45                 2             4       600    Laptop            200
  4   Male   21 Junior        CIS            Yes  ...      40                 4             6       600    Laptop            250
  5   Male   23 Senior      Other      Undecided  ...      40                 2             4       500    Laptop            100

wholesale_customers.csv: 9 columns
 Buyer/Spender Channel Region  Fresh  Milk  Grocery  Frozen  Detergent

## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 97 lines
Functions: _holm, _cohen_d, _one_sample, run_analysis


## 7. Methodology and hypotheses

Descriptive statistics, coefficients of variation, Welch tests, Mann–Whitney sensitivity, Holm correction, chi-square/Cramér's V, one-sample t and Wilcoxon tests, confidence intervals, and effect sizes.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_01_statistical_methods", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 0.77 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


shingles_data_quality.csv (2 fields)
column   dtype  missing_count  missing_percent  unique_values  constant
     A float64              0            0.000             28     False
     B float64              5           13.889             24     False

survey_data_quality.csv (14 fields)
           column   dtype  missing_count  missing_percent  unique_values  constant
               ID   int64              0              0.0             62     False
           Gender  object              0              0.0              2     False
              Age   int64              0              0.0              8     False
            Class  object              0              0.0              3     False
            Major  object              0              0.0              8     False
   Grad Intention  object              0              0.0              3     False
              GPA float64              0              0.0             16     False
       Employment  object              0     

## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'wholesale_channel_tests.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Shingle B is below the 0.35 limit by both t and Wilcoxon tests (t-test p=0.0021); Shingle A is not below the limit by the prespecified t-test (p=0.075).')

Primary evidence: wholesale_channel_tests.csv, shape=(7, 8)
         measure channel_1 channel_2  mean_difference  cohen_d  welch_p_value  mann_whitney_p_value  holm_adjusted_welch_p
           Fresh     Hotel    Retail        4571.2365   0.3663         0.0000                0.0002                 0.0001
            Milk     Hotel    Retail       -7264.7752  -1.1078         0.0000                0.0000                 0.0000
         Grocery     Hotel    Retail      -12360.7145  -1.6377         0.0000                0.0000                 0.0000
          Frozen     Hotel    Retail        2095.6390   0.4403         0.0000                0.0000                 0.0000
Detergents_Paper     Hotel    Retail       -6478.9466  -1.7590         0.0000                0.0000                 0.0000
    Delicatessen     Hotel    Retail        -337.4802  -0.1197         0.1695                0.0005                 0.1695
     total_spend     Hotel    Retail      -19775.0411  -0.8004         0.0000  

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])

## 12. Visual evidence

### Statistical Decision Evidence

![statistical_decision_evidence](../reports/figures/statistical_decision_evidence.png)

### Statistical Evidence

![statistical_evidence](../reports/figures/statistical_evidence.png)

## 13. Business interpretation

Shingle B is below the 0.35 limit by both t and Wilcoxon tests (t-test p=0.0021); Shingle A is not below the limit by the prespecified t-test (p=0.075).

The correct action is to use this result as evidence for **Prioritize category/channel investigation and decide whether moisture evidence supports a specification claim.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

The survey and shingle collection designs are undocumented; inferential results apply only under independence and sampling assumptions that cannot be fully verified.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                          artifact  size_kb       sha256
 reports/figures/statistical_decision_evidence.png    169.8 2c014d6589c5
          reports/figures/statistical_evidence.png    110.6 a9028ee56251
                              reports/metrics.json      4.4 1766e2306c23
          reports/tables/shingles_data_quality.csv      0.1 92079100c7f4
            reports/tables/survey_data_quality.csv      0.5 34138947eb31
reports/tables/survey_gender_major_contingency.csv      0.2 b60a3e58868b
      reports/tables/wholesale_channel_summary.csv      0.1 f6a67830c2a5
        reports/tables/wholesale_channel_tests.csv      1.0 33a8c80649c8
         reports/tables/wholesale_data_quality.csv      0.3 e1c0b5edf9af
       reports/tables/wholesale_region_summary.csv      0.2 61e4557f0b39


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed turn wholesale, student-survey, and shingle measurements into assumption-aware business and quality decisions. using descriptive statistics, coefficients of variation, welch tests, mann–whitney sensitivity, holm correction, chi-square/cramér's v, one-sample t and wilcoxon tests, confidence intervals, and effect sizes. The final verified conclusion is: **Shingle B is below the 0.35 limit by both t and Wilcoxon tests (t-test p=0.0021); Shingle A is not below the limit by the prespecified t-test (p=0.075).** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/01-statistical-methods/src/analysis.py
python scripts/execute_notebooks.py --project 01-statistical-methods
```